In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import os
import re

In [ ]:
device = 'cpu'

In [ ]:
def read_and_show(image_path):
    """
    :param image_path: String, path to the input image.

    Returns:
        image: PIL Image.
    """
    image = Image.open(image_path).convert('RGB')
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    return image

In [ ]:
# Load an image.
image_path = "invoice_new.png"
image = read_and_show(image_path)

In [ ]:
# Text detection model path (do not modify).
model_name = 'DB_TD500_resnet50.onnx'

In [ ]:
def ocr(image, processor, model, print_tokens=False):
    """
    :param image: PIL Image.
    :param processor: Huggingface OCR processor.
    :param model: Huggingface OCR model.
    :param print_tokens: Whether to print generated integer tokens or not

    Returns:
        generated_text: the OCR'd text string.
    """
    # We can directly perform OCR on cropped images.
    pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)
    generated_ids = model.generate(pixel_values)
    if print_tokens:
        print(generated_ids)
    generated_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0]
    return generated_text

import time

def crop_image(image, box):
    img = np.array(image)
    box = np.array(box, dtype=np.float32)

    # Ensure box[0]->box[1] is the longer (width) edge
    if np.linalg.norm(box[0] - box[1]) < np.linalg.norm(box[1] - box[2]):
        box = np.roll(box, -1, axis=0)

    # Expand box by 5% on left and right along each edge's width direction
    pad = 0.05
    top_vec = box[1] - box[0]
    bot_vec = box[2] - box[3]
    box[0] -= pad * top_vec   # left side, top corner
    box[3] -= pad * bot_vec   # left side, bottom corner
    box[1] += pad * top_vec   # right side, top corner
    box[2] += pad * bot_vec   # right side, bottom corner

    # Clamp to image bounds
    h, w = img.shape[:2]
    box[:, 0] = np.clip(box[:, 0], 0, w)
    box[:, 1] = np.clip(box[:, 1], 0, h)

    # Compute width and height of the oriented bounding box
    width = int(max(np.linalg.norm(box[0] - box[1]), np.linalg.norm(box[2] - box[3])))
    height = int(max(np.linalg.norm(box[1] - box[2]), np.linalg.norm(box[3] - box[0])))

    # Destination points for perspective transform
    dst = np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32)

    # Warp the rotated box to axis-aligned rectangle
    M = cv2.getPerspectiveTransform(box, dst)
    cropped = cv2.warpPerspective(img, M, (width, height))

    return Image.fromarray(cropped)

def infer(boxes, image, processor, model):
    start_time = time.time()
    print(f't=0: starting inference')
    generated_text = []
    for i, box in enumerate(boxes):
        print(f't={time.time() - start_time}: starting inference for box {i}')
        cropped_image = crop_image(image, box)
        print('printing cropped image:')
        plt.imshow(cropped_image)
        plt.show()
        ocr_out = ocr(cropped_image, processor, model)
        print('predicted text:')
        print(ocr_out)
        generated_text.append(ocr_out)
    print(f't={time.time() - start_time}: finished inference. Printing results:')
    for line in generated_text:
        print(line)
    return generated_text

# Function for text detection
def detect_and_show(img, td_model, show=True):
    image = img.copy()
    input_size = (1024, 1024)
    mean = (122.67891434, 116.66876762, 104.00698793)
    bin_thresh = 0.3
    poly_thresh = 0.5
    td_model.setBinaryThreshold(bin_thresh).setPolygonThreshold(poly_thresh)
    td_model.setInputParams(1.0/255, input_size, mean, True)

    # Detection.
    boxes, confs = td_model.detect(image)

    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.polylines(img, boxes, True, (0, 0, 255), 4)
    if show:
        plt.figure(figsize=(12, 9))
        plt.imshow(img[..., ::-1])
        plt.axis('off')
        plt.show()
    return boxes, confs

td_model = cv2.dnn_TextDetectionModel_DB(f"{model_name}")
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-small-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-small-printed').to(device)
boxes, confs = detect_and_show(np.array(image), td_model)
generated_text = infer(boxes, image, processor, model)

emails = []
dollar_amounts = []
for snippet in generated_text:
    TODO()

print()
print('--- ASSIGNMENT SOLUTION: ---')
print()
print('Email Ids:')
for email in emails:
    print(f'- {email}')
print()
for amount in dollar_amounts:
    print(f' Billing Amount: {amount}')